# AgriLens PoC — Data representation, leakage and next-observation NDVI

**Only** derived satellite-index forecasting is evaluated here. Dust/drought labels, crop yield and irrigation-demand ground truth are **not available**. Study site: Nile Delta, Egypt (3×3 km), 2025.

This notebook uses verified 2025 FTW crosswalk and team-supplied field-date tables. It does not redownload Sentinel-2 or assert 813 imagery is used.

In [ ]:
from pathlib import Path
import sys, json, pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from build_research_dataset import FEATURES, evaluate
audit=json.loads((ROOT / 'outputs' / 'representation_leakage_audit.json').read_text())
splits=pd.read_csv(ROOT / 'outputs' / 'sampling_frame_and_splits.csv', dtype={'field_id':'string'})
model_df=pd.read_csv(ROOT / 'outputs' / 'next_ndvi_regression.csv', parse_dates=['date','target_date'], dtype={'field_id':'string'})
print('AOI: Egypt 3x3 km | modeled fields:', splits.split_group.value_counts().to_dict())
print('Target rows:', model_df.ml_split.value_counts().to_dict())

## Leakage controls
The original pixel split had **15 overlapping field IDs**. This dataset uses distinct FTW field polygons, a 100-m minimum polygon buffer and a post-August future test window. Field-date observations still share regional weather and the same growing season; they are not fully independent crop cases.

In [ ]:
display(pd.DataFrame([{'groups':k,'nearest_polygon_distance_m':round(v,1)} for k,v in audit['geographic_minimum_pairwise_gap_m'].items()]))
assert model_df.groupby('field_id').ml_split.nunique().max() == 1
assert model_df.loc[model_df.ml_split!='test','target_date'].max() < pd.Timestamp('2025-08-01')
assert model_df.loc[model_df.ml_split=='test','date'].min() >= pd.Timestamp('2025-08-01')
print('Field and time holdouts: passed')

## Representation and index trajectory
The original 982 field-date records were filtered to 707 field-date records for 23 model-predicted fields. Three additional buffer fields are excluded here, leaving 20. Field 1665168 is excluded because its full predicted polygon is approximately 132 km², likely merging multiple real fields.

In [ ]:
display(splits.groupby('split_group').agg(fields=('field_id','size'),area_ha=('area_ha','sum')).round(2))
fig,ax=plt.subplots(figsize=(8,3.5))
for fid in model_df.loc[model_df.ml_split=='test','field_id'].drop_duplicates().iloc[:3]:
    s=model_df[model_df.field_id==fid]
    ax.plot(s.date,s.ndvi_mean,marker='.',label=str(fid))
ax.set_ylabel('Observed field mean NDVI');ax.set_xlabel('Observation date');ax.set_title('Three held-out eastern fields — current-index trajectories');ax.legend();fig.autofmt_xdate();plt.show()

## Exploratory regression versus persistence
The target is the field mean NDVI at the **next accepted Sentinel-2 observation** 3–18 days later. The model uses only features available on the current observation date. Hyperparameter choice uses geographic validation fields and the frozen model is scored on separate fields in August–September. A simple persistence predictor (NDVI(t+1)=NDVI(t)) is the benchmark.

In [ ]:
scores,preds=evaluate(model_df)
summary=pd.DataFrame([{'split':s,'n_pairs':scores[s]['ridge']['rows'], 'ridge_MAE':scores[s]['ridge']['MAE'], 'persistence_MAE':scores[s]['persistence']['MAE'], 'ridge_R2':scores[s]['ridge']['R2']} for s in ['train','validation','test']])
display(summary.round(4))
assert scores['test']['ridge']['MAE'] > scores['test']['persistence']['MAE']
print('Conclusion: Ridge DOES NOT outperform persistence on the future+geographical holdout.')

## Interpretation and next independent evidence
The held-out test includes only 5 fields and 37 field-date pairs from 9 dates. This is a **preliminary satellite-index forecasting experiment**. It says nothing about causal dust damage or verified drought stress. Obtain agronomist/field observations, crop type, irrigation records and multiple locations/seasons before interpreting stress causes or claiming agronomic accuracy.